## Centerline

In [ ]:
import numpy
import sys
from scipy.ndimage import distance_transform_edt
from skimage.feature import peak_local_max
from scipy.spatial import cKDTree
import vtk

sys.path.insert(0, "../src")  # add Folder_2 path to search list

from meshing_functions import getSurfaceMesh, tetra_mesh_from_stl, tetra_shell_from_two_surfaces

pixel_spacing = numpy.loadtxt("/Users/skardova/Documents/MRI_data/2026-Eyes-project/Trial2/VTI_0.5_fat_AIMax/pixel_spacing.txt")


pixel_spacing = numpy.loadtxt("/Users/skardova/Documents/MRI_data/2026-Eyes-project/Trial2/VTI_0.5_fat_AIMax/pixel_spacing.txt")

base = "/Users/skardova/Documents/MRI_data/2026-Eyes-project/Trial2/cropped T1/Meshes/"

filename_base = "segmentations_pdt1_v160725._filtered_surf_" # + _id=1.npy



## 1.  Load mask

In [2]:
id = 1

mask = numpy.load(base + filename_base + "id=" + str(id) +".npy")

dist = distance_transform_edt(mask)
center_pts = peak_local_max(dist, footprint=numpy.ones((3,3,3)), labels=mask)

print(len(center_pts))

40


## 2.  Detect centerpointa + order

In [3]:
points = center_pts.astype(float)

tree = cKDTree(points)

ordered = [0]
visited = {0}

for _ in range(len(points)-1):
    current = ordered[-1]

    dists, idxs = tree.query(points[current], k=10)

    for idx in idxs[1:]:
        if idx not in visited:
            ordered.append(idx)
            visited.add(idx)
            break

ordered_points = points[ordered]

## 3.  Write file

In [ ]:
vtk_points = vtk.vtkPoints()

for p in ordered_points:
    vtk_points.InsertNextPoint(float(p[0]*pixel_spacing[0]), float(p[1]*pixel_spacing[0]), float(p[2]*pixel_spacing[0]))

polyline = vtk.vtkPolyLine()
polyline.GetPointIds().SetNumberOfIds(len(ordered_points))

for i in range(len(ordered_points)):
    polyline.GetPointIds().SetId(i, i)

cells = vtk.vtkCellArray()
cells.InsertNextCell(polyline)

polydata = vtk.vtkPolyData()
polydata.SetPoints(vtk_points)
polydata.SetLines(cells)

writer = vtk.vtkXMLPolyDataWriter()
writer.SetFileName(base + "centerline_"+str(id)+".vtp")
writer.SetInputData(polydata)
writer.Write()

1